# Malaria Parasite Detection — YOLOv8n Training
**Runtime → Change runtime type → T4 GPU** before running.

Steps:
1. Install dependencies
2. Mount Google Drive and unzip dataset
3. Train YOLOv8n with stronger settings for the imbalanced dataset
4. Evaluate on the validation set
5. Save `best.pt` back to Drive

Notes:
- The dataset is heavily imbalanced, so the rare classes need more training signal.
- Use the updated training settings below instead of the older generic defaults.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Cell 2: Install ultralytics ───────────────────────────────────────────────
!pip install -q ultralytics
from ultralytics import YOLO
print('ultralytics ready')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# ── Cell 4: Set the path to your zip in Drive ─────────────────────────────────
# UPDATE THIS if you put the zip in a subfolder, e.g. 'MyDrive/malaria/...'
ZIP_PATH    = '/content/drive/MyDrive/malaria_yolo_dataset.zip'
DATASET_DIR = '/content/yolo_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/malaria_model'   # best.pt saved here

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Zip: {ZIP_PATH}')
print(f'Dataset will extract to: {DATASET_DIR}')
print(f'Model output: {OUTPUT_DIR}')

In [ ]:
# ── Cell 5: Unzip dataset ─────────────────────────────────────────────────────
import zipfile, os

if not os.path.exists(DATASET_DIR):
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Done.')
else:
    print('Dataset already extracted.')

train_imgs = len(os.listdir(f'{DATASET_DIR}/images/train'))
val_imgs   = len(os.listdir(f'{DATASET_DIR}/images/val'))
print(f'Train: {train_imgs} images   Val: {val_imgs} images')

In [ ]:
# ── Cell 6: Write data.yaml with correct Colab paths ─────────────────────────
import yaml

data_cfg = {
    'path':  DATASET_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    6,
    'names': {
        0: 'red blood cell',
        1: 'trophozoite',
        2: 'ring',
        3: 'schizont',
        4: 'gametocyte',
        5: 'leukocyte',
    }
}

yaml_path = f'{DATASET_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, allow_unicode=True)

print(f'data.yaml written to {yaml_path}')
print(open(yaml_path).read())

In [ ]:
# ── Cell 7: Train YOLOv8n with stronger settings for the imbalanced dataset ────
# On a T4 GPU this takes longer, but the extra epochs and gentler augmentation
# usually help the minority classes learn better.

model = YOLO('yolov8n.pt')   # downloads pretrained backbone automatically

results = model.train(
    data         = yaml_path,
    epochs       = 100,
    batch        = 16,          # smaller batch is safer with larger images
    imgsz        = 768,         # larger image size helps small parasite objects
    device       = 0,           # GPU
    project      = '/content/runs',
    name         = 'malaria',
    exist_ok     = True,
    patience     = 25,
    save_period  = 10,
    workers      = 4,
    cache        = True,
    cos_lr       = True,
    close_mosaic = 10,
    optimizer    = 'AdamW',
    lr0          = 0.003,
    weight_decay = 0.0005,
    # ── augmentation ──────────────────────────────────────────────────────────
    hsv_h        = 0.01,
    hsv_s        = 0.5,
    hsv_v        = 0.4,
    degrees      = 8.0,
    translate    = 0.08,
    scale        = 0.4,
    flipud       = 0.2,
    fliplr       = 0.5,
    mosaic       = 0.7,
    mixup        = 0.0,
    copy_paste   = 0.0,
)

print('Training complete!')
print(f'Best model saved to: {results.save_dir}/weights/best.pt')

In [ ]:
# ── Cell 8: Print final metrics ───────────────────────────────────────────────
if hasattr(results, 'results_dict'):
    rd = results.results_dict
    print('=' * 50)
    print('  FINAL VALIDATION METRICS')
    print('=' * 50)
    print(f"  Precision  : {rd.get('metrics/precision(B)', 0):.4f}")
    print(f"  Recall     : {rd.get('metrics/recall(B)', 0):.4f}")
    print(f"  mAP@50     : {rd.get('metrics/mAP50(B)', 0):.4f}")
    print(f"  mAP@50-95  : {rd.get('metrics/mAP50-95(B)', 0):.4f}")
    print('=' * 50)

In [ ]:
# ── Cell 9: Save best.pt to Google Drive ─────────────────────────────────────
import shutil
from pathlib import Path
 
best_src = Path(results.save_dir) / 'weights' / 'best.pt'
best_dst = Path(OUTPUT_DIR) / 'best.pt'

shutil.copy2(best_src, best_dst)
size_mb = best_dst.stat().st_size / 1_048_576
print(f'Saved: {best_dst}  ({size_mb:.1f} MB)')
print()
print('Next: download best.pt from Google Drive to:')
print('  vision-backend/models/best.pt')

In [ ]:
# ── Cell 10 (optional): Per-class metrics on val set ─────────────────────────
CLASS_NAMES = ['red blood cell','trophozoite','ring','schizont','gametocyte','leukocyte']

val_results = model.val(data=yaml_path, split='val', verbose=False)

print(f"{'CLASS':<18} {'P':>6} {'R':>6} {'mAP50':>7} {'mAP50-95':>9}")
print('-' * 50)
for i, name in enumerate(CLASS_NAMES):
    p   = float(val_results.box.p[i])    if i < len(val_results.box.p)    else 0
    r   = float(val_results.box.r[i])    if i < len(val_results.box.r)    else 0
    a50 = float(val_results.box.ap50[i]) if i < len(val_results.box.ap50) else 0
    a   = float(val_results.box.ap[i])   if i < len(val_results.box.ap)   else 0
    print(f'{name:<18} {p:>6.3f} {r:>6.3f} {a50:>7.3f} {a:>9.3f}')

worst_recall = min(float(val_results.box.r[i]) for i in range(len(CLASS_NAMES)) if i < len(val_results.box.r))
weak_classes = [
    name
    for i, name in enumerate(CLASS_NAMES)
    if i < len(val_results.box.r) and float(val_results.box.r[i]) < 0.20
]

print('-' * 50)
print(f'Worst recall: {worst_recall:.3f}')
if weak_classes:
    print('Weak classes:')
    for name in weak_classes:
        print(f'  - {name}')
    print('Recommendation: train longer or clean/balance the rare classes before trusting the model.')